In [ ]:
# ===============================
# COMPLETE RAM + GPU CLEANUP
# ===============================

import gc
import torch
import os

print("Cleaning RAM and GPU memory...")

# Delete unused variables safely
for var in list(globals()):
    if var not in [
        "os",
        "gc",
        "torch"
    ]:
        del globals()[var]

# Python Garbage Collection
gc.collect()

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("✅ RAM & GPU Memory Cleaned")

Cleaning RAM and GPU memory...
✅ RAM & GPU Memory Cleaned


In [ ]:
# =====================================================
# INSTALL
# =====================================================
!pip -q install transformers librosa soundfile scikit-learn xgboost tqdm

import os
import gc
import torch
import librosa
import numpy as np
from tqdm import tqdm
from google.colab import drive

from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier


# =====================================================
# MOUNT DRIVE (UNCHANGED PATH)
# =====================================================
drive.mount('/content/drive')

TRAIN_PATH="/content/drive/MyDrive/TamilDialect/Train"
TEST_PATH="/content/drive/MyDrive/TamilDialect/Test"
OUTPUT_TXT="/content/drive/MyDrive/TamilDialect/claassiferfinal.txt"


# =====================================================
# DEVICE
# =====================================================
device="cuda" if torch.cuda.is_available() else "cpu"
print("Using:",device)


# =====================================================
# LOAD WAV2VEC2
# =====================================================
feature_extractor=Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/wav2vec2-base"
)

model=Wav2Vec2Model.from_pretrained(
    "facebook/wav2vec2-base"
).to(device)

model.eval()


# =====================================================
# FEATURE EXTRACTION
# =====================================================
def extract_feature(path):

    audio,_=librosa.load(path,sr=16000)

    inputs=feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        out=model(inputs.input_values.to(device))

    hidden=out.last_hidden_state

    emb=torch.cat([
        hidden.mean(dim=1),
        hidden.std(dim=1)
    ],dim=1)

    return emb.squeeze().cpu().numpy()


# =====================================================
# TRAIN FEATURE EXTRACTION
# =====================================================
print("\nExtracting TRAIN features")

X=[]
y=[]

for dialect in os.listdir(TRAIN_PATH):

    dpath=os.path.join(TRAIN_PATH,dialect)
    if not os.path.isdir(dpath):
        continue

    print("Dialect:",dialect)

    for root,_,files in os.walk(dpath):

        for f in files:
            if f.endswith(".wav"):

                feat=extract_feature(os.path.join(root,f))

                X.append(feat)
                y.append(dialect)

                if len(X)%40==0:
                    gc.collect()
                    torch.cuda.empty_cache()

X=np.array(X)
y=np.array(y)

print("Samples:",len(X))


# =====================================================
# LABEL ENCODING
# =====================================================
encoder=LabelEncoder()
y_enc=encoder.fit_transform(y)


# =====================================================
# CLASSIFIER (HIGH ACCURACY)
# =====================================================
print("\nTraining classifier")

clf=make_pipeline(
    StandardScaler(),
    XGBClassifier(
        n_estimators=1000,
        max_depth=12,
        learning_rate=0.02,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1
    )
)

clf.fit(X,y_enc)

print("✅ Training Done")


# =====================================================
# TEST PREDICTION
# =====================================================
print("\nPredicting TEST")

results=[]

for f in tqdm(sorted(os.listdir(TEST_PATH))):

    if not f.endswith(".wav"):
        continue

    feat=extract_feature(os.path.join(TEST_PATH,f))

    pred=clf.predict([feat])[0]
    label=encoder.inverse_transform([pred])[0]

    results.append((f,label))


# =====================================================
# SAVE OUTPUT
# =====================================================
with open(OUTPUT_TXT,"w") as out:
    for f,l in results:
        out.write(f"{f} -> {l}\n")

print("✅ Saved:",OUTPUT_TXT)


# =====================================================
# CLEAN RAM
# =====================================================
gc.collect()
torch.cuda.empty_cache()

print("\n✅ FINISHED")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using: cuda


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Extracting TRAIN features
Dialect: Central_Dialect
Dialect: Southern_Dialect
Dialect: Northern_Dialect
Dialect: Western_Dialect
Samples: 5134

Training classifier
✅ Training Done

Predicting TEST


100%|██████████| 579/579 [00:52<00:00, 10.95it/s]


✅ Saved: /content/drive/MyDrive/TamilDialect/claassiferfinal.txt

✅ FINISHED
